In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
DATA_PATH = '/content/drive/MyDrive/software-build-failure-prediction/data/raw/final-2017-01-25.csv.gz'
print(os.path.exists(DATA_PATH))

True


In [3]:
import pandas as pd

chunks = []
for chunk in pd.read_csv(DATA_PATH, compression='gzip', chunksize=50_000, low_memory=False):
    filtered = chunk[(chunk['gh_lang'] == 'ruby') & (chunk['tr_status'].isin(['passed', 'failed']))]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)
df.head()

(1542248, 66)


,tr_build_id,gh_project_name,gh_is_pr,gh_pr_created_at,gh_pull_req_num,gh_lang,git_merged_with,git_branch,gh_num_commits_in_push,gh_commits_in_push,...,tr_log_num_test_suites_run,tr_log_num_test_suites_ok,tr_log_num_test_suites_failed,tr_log_tests_failed,tr_log_testduration,tr_log_buildduration,tr_original_commit,tr_duration,tr_status,tr_jobs
0,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,NaN,NaN,NaN,NaN,114.83,NaN,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]"
1,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,NaN,NaN,NaN,NaN,171.37,NaN,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]"
2,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,NaN,NaN,NaN,NaN,177.02,NaN,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]"
3,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,NaN,NaN,NaN,NaN,182.05,NaN,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]"
4,3154,rspec/rspec-core,False,NaN,NaN,ruby,NaN,master,NaN,NaN,...,NaN,NaN,NaN,NaN,113.26,NaN,029e6972fcf719542deff1b2619d2945146e84da,956.0,passed,"[3161, 3163, 3160, 3162, 3164]"


In [4]:
print(df.columns.tolist())

['tr_build_id', 'gh_project_name', 'gh_is_pr', 'gh_pr_created_at', 'gh_pull_req_num', 'gh_lang', 'git_merged_with', 'git_branch', 'gh_num_commits_in_push', 'gh_commits_in_push', 'git_prev_commit_resolution_status', 'git_prev_built_commit', 'tr_prev_build', 'gh_first_commit_created_at', 'gh_team_size', 'git_all_built_commits', 'git_num_all_built_commits', 'git_trigger_commit', 'tr_virtual_merged_into', 'gh_num_issue_comments', 'gh_num_commit_comments', 'gh_num_pr_comments', 'git_diff_src_churn', 'git_diff_test_churn', 'gh_diff_files_added', 'gh_diff_files_deleted', 'gh_diff_files_modified', 'gh_diff_tests_added', 'gh_diff_tests_deleted', 'gh_diff_src_files', 'gh_diff_doc_files', 'gh_diff_other_files', 'gh_num_commits_on_files_touched', 'gh_sloc', 'gh_test_lines_per_kloc', 'gh_test_cases_per_kloc', 'gh_asserts_cases_per_kloc', 'gh_by_core_team_member', 'gh_description_complexity', 'gh_pushed_at', 'gh_build_started_at', 'gh_repo_age', 'gh_repo_num_commits', 'tr_job_id', 'tr_build_number',

In [5]:
# Collapse to one row per build (tr_build_id)
# Pre-execution features are identical across jobs of the same build, so take the first
# Post-execution log features: aggregate (e.g., max tests_failed across jobs = build fails if any job fails)

agg_dict = {col: 'first' for col in df.columns if col not in
            ['tr_log_num_tests_failed', 'tr_log_num_tests_ok', 'tr_log_num_tests_run',
             'tr_log_bool_tests_failed', 'tr_log_testduration', 'tr_log_buildduration']}

# For the log/test columns, aggregate instead of just taking first job's value
agg_dict.update({
    'tr_log_num_tests_failed': 'sum',
    'tr_log_num_tests_ok': 'sum',
    'tr_log_num_tests_run': 'sum',
    'tr_log_bool_tests_failed': 'max',
    'tr_log_testduration': 'sum',
    'tr_log_buildduration': 'sum',
})

df_build_level = df.groupby('tr_build_id', as_index=False).agg(agg_dict)
print(df_build_level.shape)

(252151, 66)


In [6]:
# Save the build-level aggregated dataset
save_path = '/content/drive/MyDrive/software-build-failure-prediction/data/processed/travistorrent_ruby_builds.csv'
df_build_level.to_csv(save_path, index=False)
print('Saved:', df_build_level.shape)

# Check class balance — this determines if you need SMOTE/class weighting later
print(df_build_level['tr_status'].value_counts())
print(df_build_level['tr_status'].value_counts(normalize=True) * 100)

Saved: (252151, 66)
tr_status
passed    197850
failed     54301
Name: count, dtype: int64
tr_status
passed    78.464888
failed    21.535112
Name: proportion, dtype: float64
